# Model Comparison: xLSTM Models

This notebook evaluates and compares four music generation models using various metrics:
1. **Musical Quality Metrics**: Pitch range, key stability, note density, etc.
2. **Self-Similarity Matrix (SSM) Metrics**: Structure analysis.
3. **Similarity Error (SE)**: Comparison against real validation data.

## Models
- **Model 1**: `/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260208_041715-xlstm_lmd_512d_2048ctx_12b-checkpoint-84000`
- **Model 2**: `/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260128_191749`
- **Model 3**: `/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260225_155134-xlstm_lmd_512d_4096ctx_12b-checkpoint-84000`
- **Model 4**: `/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-4-recurrent-state/single-shot-generation/results/gen1-before-preprocess/midi`
- **Model 5**: `/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-4-recurrent-state/single-shot-generation/results/gen-after-preprocess/midi`



- **Model 1**: Before Preprocessing 512d 2048ctx - checkpoint 84000
- **Model 2**: before preprocessing - ??
- **Model 3**: before preprocessing - 512d 4096ctx - checkpoint 84000
- **Model 4**: before preprocessing - single-shot-generation
- **Model 5**: after preprocessing

In [1]:
import os
import pandas as pd
from pathlib import Path
import json

# Define Paths
BASE_DIR = Path("/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation")
SCRIPTS_DIR = BASE_DIR
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

MODEL_1_PATH = Path("/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260208_041715-xlstm_lmd_512d_2048ctx_12b-checkpoint-84000")
MODEL_2_PATH = Path("/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260128_191749")
MODEL_3_PATH = Path("/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260225_155134-xlstm_lmd_512d_4096ctx_12b-checkpoint-84000")
MODEL_4_PATH = Path("/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-4-recurrent-state/single-shot-generation/results/gen1-before-preprocess/midi")
MODEL_5_PATH = Path("/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-4-recurrent-state/single-shot-generation/results/gen-after-preprocess/midi")
MODEL_LOOKBACK_RNN_PATH = Path("/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/melody_rnn/output/generated_samples_for_eval")
MODEL_MUSEFORMER_PATH = Path("/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/museformer_output")
MODEL_VALIDATION_PATH = Path("/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/validation_sample_100")
MODEL_6_PATH = Path("/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-4-recurrent-state/single-shot-generation/results/gen-after-preprocess-24k-dataset/midi")

# Validation Data for SE
REAL_LIST = Path("/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt")
REAL_DIR = Path("/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/museformer_baseline/29k_backup/final_output")

# Python Executable (muspy-env)
PYTHON_EXEC = "/home/e20365/miniconda3/envs/muspy-env/bin/python"

print(f"Model 1: {MODEL_1_PATH}")
print(f"Model 2: {MODEL_2_PATH}")
print(f"Model 3: {MODEL_3_PATH}")
print(f"Model 4: {MODEL_4_PATH}")
print(f"Model 5: {MODEL_5_PATH}")
print(f"Lookback RNN: {MODEL_LOOKBACK_RNN_PATH}")
print(f"Museformer: {MODEL_MUSEFORMER_PATH}")
print(f"Validation/Human: {MODEL_VALIDATION_PATH}")
print(f"xLSTM (24k Dataset): {MODEL_6_PATH}")

Model 1: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260208_041715-xlstm_lmd_512d_2048ctx_12b-checkpoint-84000
Model 2: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260128_191749
Model 3: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260225_155134-xlstm_lmd_512d_4096ctx_12b-checkpoint-84000
Model 4: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-4-recurrent-state/single-shot-generation/results/gen1-before-preprocess/midi
Model 5: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-4-recurrent-state/single-shot-generation/results/gen-after-preprocess/midi
Lookback RNN: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/melody_rnn/output/generated_samples_for_eval
Museformer: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/

## 1. Musical Quality Metrics

In [2]:
def run_musical_metrics(model_path, label):
    print(f"--- Running Musical Metrics for {label} ---")
    out_csv = RESULTS_DIR / f"musical_metrics_{label}.csv"
    summary_csv = RESULTS_DIR / f"musical_summary_{label}.csv"
    
    # 1. Compute Metrics
    cmd = f'{PYTHON_EXEC} {SCRIPTS_DIR}/musical_quality_metrics.py --input_dir "{model_path}" --out_csv "{out_csv}"'
    print(f"Executing: {cmd}")
    ret = os.system(cmd)
    if ret != 0: print(f"Error running musical metrics for {label}")
        
    # 2. Aggregate
    cmd = f'{PYTHON_EXEC} {SCRIPTS_DIR}/aggregate_metrics.py --input_csv "{out_csv}" --output_csv "{summary_csv}"'
    print(f"Executing: {cmd}\n")
    ret = os.system(cmd)
    if ret != 0: print(f"Error aggregating metrics for {label}")
        
run_musical_metrics(MODEL_1_PATH, "model1")
run_musical_metrics(MODEL_2_PATH, "model2")
run_musical_metrics(MODEL_3_PATH, "model3")
run_musical_metrics(MODEL_4_PATH, "model4")
run_musical_metrics(MODEL_5_PATH, "model5")
run_musical_metrics(MODEL_LOOKBACK_RNN_PATH, "lookback_rnn")
run_musical_metrics(MODEL_MUSEFORMER_PATH, "museformer_set1")
run_musical_metrics(MODEL_VALIDATION_PATH, "validation")
run_musical_metrics(MODEL_6_PATH, "model6")

--- Running Musical Metrics for model1 ---
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/musical_quality_metrics.py --input_dir "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260208_041715-xlstm_lmd_512d_2048ctx_12b-checkpoint-84000" --out_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_model1.csv"


[OK] Wrote 80 rows -> /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_model1.csv


Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/aggregate_metrics.py --input_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_model1.csv" --output_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_summary_model1.csv"



[INFO] Loaded 80 rows from /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_model1.csv
[INFO] Rows after filtering errors: 80

Metrics Summary (Mean ± Std):
------------------------------------------------------------
Metric                    | Formatted                     
------------------------------------------------------------
bars_N                    | 31.5875 ± 19.2398             
key_stability             | 0.5211 ± 0.2855               
key_change_rate           | 0.4797 ± 0.3058               
chord_diversity           | 0.4283 ± 0.2168               
pitch_range               | 38.0000 ± 19.5221             
stepwise_ratio            | 0.5308 ± 0.2282               
note_density_mean         | 31.7950 ± 18.3028             
note_density_std          | 11.2843 ± 8.3111              
duration_entropy          | 1.0289 ± 0.5807               
[INFO] Summary saved to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fy

[OK] Wrote 60 rows -> /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_model2.csv


...

Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/aggregate_metrics.py --input_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_validation.csv" --output_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_summary_validation.csv"



[INFO] Loaded 100 rows from /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_validation.csv
[INFO] Rows after filtering errors: 100

Metrics Summary (Mean ± Std):
------------------------------------------------------------
Metric                    | Formatted                     
------------------------------------------------------------
bars_N                    | 93.6100 ± 31.3010             
key_stability             | 0.3577 ± 0.1973               
key_change_rate           | 0.6976 ± 0.1826               
chord_diversity           | 0.3010 ± 0.1269               
pitch_range               | 48.8000 ± 10.7825             
stepwise_ratio            | 0.3511 ± 0.1908               
note_density_mean         | 39.5787 ± 17.1004             
note_density_std          | 11.5049 ± 7.2178              
duration_entropy          | 1.2629 ± 0.4098               
[INFO] Summary saved to /scratch1/e20-fyp-xlstm-music-generation/e20fypte

[OK] Wrote 120 rows -> /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_model6.csv


Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/aggregate_metrics.py --input_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_model6.csv" --output_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_summary_model6.csv"



[INFO] Loaded 120 rows from /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/musical_metrics_model6.csv
[INFO] Rows after filtering errors: 120

Metrics Summary (Mean ± Std):
------------------------------------------------------------
Metric                    | Formatted                     
------------------------------------------------------------
bars_N                    | 60.6917 ± 57.0797             
key_stability             | 0.3653 ± 0.2333               
key_change_rate           | 0.5649 ± 0.2131               
chord_diversity           | 0.5592 ± 0.2048               
pitch_range               | 51.3417 ± 17.4638             
stepwise_ratio            | 0.4571 ± 0.2057               
note_density_mean         | 27.4110 ± 15.0262             
note_density_std          | 9.9759 ± 7.1155               
duration_entropy          | 1.2532 ± 0.4077               
[INFO] Summary saved to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/

## 2. Structure Metrics (SSM)

In [3]:
def run_ssm_metrics(model_path, label):
    print(f"--- Running SSM Metrics for {label} ---")
    ssm_out_dir = RESULTS_DIR / f"ssm_outputs_{label}"
    metrics_csv = RESULTS_DIR / f"ssm_metrics_{label}.csv"
    summary_csv = RESULTS_DIR / f"ssm_summary_{label}.csv"
    
    # 1. Compute SSMs
    cmd = f'{PYTHON_EXEC} {SCRIPTS_DIR}/compute_bar_ssm.py --input_dir "{model_path}" --output_dir "{ssm_out_dir}" --feature chroma --binarize'
    print(f"Executing: {cmd}")
    ret = os.system(cmd)
    if ret != 0: print(f"Error computing SSMs for {label}")
        
    # 2. Compute Metrics from SSMs
    cmd = f'{PYTHON_EXEC} {SCRIPTS_DIR}/compute_metrics_from_ssm.py --ssm_dir "{ssm_out_dir}" --out_csv "{metrics_csv}"'
    print(f"Executing: {cmd}")
    ret = os.system(cmd)
    if ret != 0: print(f"Error computing SSM metrics for {label}")
        
    # 3. Aggregate
    cmd = f'{PYTHON_EXEC} {SCRIPTS_DIR}/aggregate_metrics.py --input_csv "{metrics_csv}" --output_csv "{summary_csv}"'
    print(f"Executing: {cmd}\n")
    ret = os.system(cmd)
    if ret != 0: print(f"Error aggregating SSM metrics for {label}")

run_ssm_metrics(MODEL_1_PATH, "model1")
run_ssm_metrics(MODEL_2_PATH, "model2")
run_ssm_metrics(MODEL_3_PATH, "model3")
run_ssm_metrics(MODEL_4_PATH, "model4")
run_ssm_metrics(MODEL_5_PATH, "model5")
run_ssm_metrics(MODEL_LOOKBACK_RNN_PATH, "lookback_rnn")
run_ssm_metrics(MODEL_MUSEFORMER_PATH, "museformer_set1")
run_ssm_metrics(MODEL_VALIDATION_PATH, "validation")
run_ssm_metrics(MODEL_6_PATH, "model6")

--- Running SSM Metrics for model1 ---
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/compute_bar_ssm.py --input_dir "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260208_041715-xlstm_lmd_512d_2048ctx_12b-checkpoint-84000" --output_dir "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/ssm_outputs_model1" --feature chroma --binarize


[INFO] Found 80 MIDI files under: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260208_041715-xlstm_lmd_512d_2048ctx_12b-checkpoint-84000
[INFO] Output dir: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/ssm_outputs_model1
[INFO] Settings: fs=50, feature=chroma, binarize=True
[OK] temp_0.6_bars_16_001.mid -> bars=29 | saved: temp_0.6_bars_16_001__barSSM_chroma_fs50.npy, temp_0.6_bars_16_001__barSSM_chroma_fs50.png
[OK] temp_0.6_bars_16_002.mid -> bars=16 | saved: temp_0.6_bars_16_002__barSSM_chroma_fs50.npy, temp_0.6_bars_16_002__barSSM_chroma_fs50.png
[OK] temp_0.6_bars_16_003.mid -> bars=20 | saved: temp_0.6_bars_16_003__barSSM_chroma_fs50.npy, temp_0.6_bars_16_003__barSSM_chroma_fs50.png
[OK] temp_0.6_bars_16_004.mid -> bars=19 | saved: temp_0.6_bars_16_004__barSSM_chroma_fs50.npy, temp_0.6_bars_16_004__barSSM_chroma_fs50.png
[OK] temp_0.6_bars_16_005.mid -> bars=17 | saved: temp_0.6_bars_1

[OK] temp_0.8_bars_64_001.mid -> bars=51 | saved: temp_0.8_bars_64_001__barSSM_chroma_fs50.npy, temp_0.8_bars_64_001__barSSM_chroma_fs50.png
[OK] temp_0.8_bars_64_002.mid -> bars=65 | saved: temp_0.8_bars_64_002__barSSM_chroma_fs50.npy, temp_0.8_bars_64_002__barSSM_chroma_fs50.png
[OK] temp_0.8_bars_64_003.mid -> bars=64 | saved: temp_0.8_bars_64_003__barSSM_chroma_fs50.npy, temp_0.8_bars_64_003__barSSM_chroma_fs50.png
[OK] temp_0.8_bars_64_004.mid -> bars=36 | saved: temp_0.8_bars_64_004__barSSM_chroma_fs50.npy, temp_0.8_bars_64_004__barSSM_chroma_fs50.png
[OK] temp_0.8_bars_64_005.mid -> bars=63 | saved: temp_0.8_bars_64_005__barSSM_chroma_fs50.npy, temp_0.8_bars_64_005__barSSM_chroma_fs50.png
[OK] temp_0.9_bars_16_001.mid -> bars=16 | saved: temp_0.9_bars_16_001__barSSM_chroma_fs50.npy, temp_0.9_bars_16_001__barSSM_chroma_fs50.png
[OK] temp_0.9_bars_16_002.mid -> bars=17 | saved: temp_0.9_bars_16_002__barSSM_chroma_fs50.npy, temp_0.9_bars_16_002__barSSM_chroma_fs50.png
[OK] temp_0.9

Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/compute_metrics_from_ssm.py --ssm_dir "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/ssm_outputs_model1" --out_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/ssm_metrics_model1.csv"


/home/e20365/miniconda3/envs/muspy-env/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/home/e20365/miniconda3/envs/muspy-env/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/home/e20365/miniconda3/envs/muspy-env/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


...

/home/e20365/miniconda3/envs/muspy-env/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (3) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


/home/e20365/miniconda3/envs/muspy-env/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


[OK] Wrote 120 rows -> /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/ssm_metrics_model6.csv


Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/aggregate_metrics.py --input_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/ssm_metrics_model6.csv" --output_csv "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/ssm_summary_model6.csv"



[INFO] Loaded 120 rows from /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/ssm_metrics_model6.csv

Metrics Summary (Mean ± Std):
------------------------------------------------------------
Metric                    | Formatted                     
------------------------------------------------------------
bars_N                    | 60.6917 ± 57.0797             
avg_offdiag               | 0.5015 ± 0.1662               
rep_density               | 0.2858 ± 0.2003               
struct_entropy            | 6.5158 ± 1.5908               
block_coherence           | 0.4233 ± 0.1785               
[INFO] Summary saved to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/ssm_summary_model6.csv


## 3. Similarity Error (SE)

In [4]:
def run_se_metric(model_path, label):
    print(f"--- Running SE Metric for {label} ---")
    out_json = RESULTS_DIR / f"se_results_{label}.json"
    
    cmd = f'{PYTHON_EXEC} {SCRIPTS_DIR}/compute_similarity_error.py --real_list "{REAL_LIST}" --real_dir "{REAL_DIR}" --gen_dir "{model_path}" --out "{out_json}" --limit 100'
    print(f"Executing: {cmd}\n")
    ret = os.system(cmd)
    if ret != 0: print(f"Error computing SE for {label}")

run_se_metric(MODEL_1_PATH, "model1")
run_se_metric(MODEL_2_PATH, "model2")
run_se_metric(MODEL_3_PATH, "model3")
run_se_metric(MODEL_4_PATH, "model4")
run_se_metric(MODEL_5_PATH, "model5")
run_se_metric(MODEL_LOOKBACK_RNN_PATH, "lookback_rnn")
run_se_metric(MODEL_MUSEFORMER_PATH, "museformer_set1")
run_se_metric(MODEL_VALIDATION_PATH, "validation")
run_se_metric(MODEL_6_PATH, "model6")

--- Running SE Metric for model1 ---
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/compute_similarity_error.py --real_list "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt" --real_dir "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/museformer_baseline/29k_backup/final_output" --gen_dir "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260208_041715-xlstm_lmd_512d_2048ctx_12b-checkpoint-84000" --out "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/se_results_model1.json" --limit 100



[INFO] Loading real MIDIs from list: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt (limit=100)
[INFO] Listing generated MIDIs from: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260208_041715-xlstm_lmd_512d_2048ctx_12b-checkpoint-84000
[INFO] Found 100 real, 80 generated files.
[INFO] Computing curve for Real Data...
[INFO] Computing curve for Generated Data...
Similarity Error (SE): 0.063880
[INFO] Saved results to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/se_results_model1.json
--- Running SE Metric for model2 ---
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/compute_similarity_error.py --real_list "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt" --real_dir "/scratch1/e20-fyp-xlstm

[INFO] Loading real MIDIs from list: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt (limit=100)
[INFO] Listing generated MIDIs from: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260128_191749
[INFO] Found 100 real, 60 generated files.
[INFO] Computing curve for Real Data...
[INFO] Computing curve for Generated Data...
Similarity Error (SE): 0.135181
[INFO] Saved results to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/se_results_model2.json
--- Running SE Metric for model3 ---
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/compute_similarity_error.py --real_list "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt" --real_dir "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/d

[INFO] Loading real MIDIs from list: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt (limit=100)
[INFO] Listing generated MIDIs from: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-2/generated_batch_20260225_155134-xlstm_lmd_512d_4096ctx_12b-checkpoint-84000
[INFO] Found 100 real, 80 generated files.
[INFO] Computing curve for Real Data...
[INFO] Computing curve for Generated Data...
Similarity Error (SE): 0.093370
[INFO] Saved results to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/se_results_model3.json
--- Running SE Metric for model4 ---
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/compute_similarity_error.py --real_list "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt" --real_dir "/scratch1/e20-fyp-xlstm

[INFO] Loading real MIDIs from list: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt (limit=100)
[INFO] Listing generated MIDIs from: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-4-recurrent-state/single-shot-generation/results/gen1-before-preprocess/midi
[INFO] Found 100 real, 120 generated files.
[INFO] Computing curve for Real Data...
[INFO] Computing curve for Generated Data...
Similarity Error (SE): 0.067280
[INFO] Saved results to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/se_results_model4.json
--- Running SE Metric for model5 ---
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/compute_similarity_error.py --real_list "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt" --real_dir "/scratch1/e20-fyp-xlstm

[INFO] Loading real MIDIs from list: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt (limit=100)
[INFO] Listing generated MIDIs from: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-4-recurrent-state/single-shot-generation/results/gen-after-preprocess/midi
[INFO] Found 100 real, 120 generated files.
[INFO] Computing curve for Real Data...
[INFO] Computing curve for Generated Data...
Similarity Error (SE): 0.142355
[INFO] Saved results to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/se_results_model5.json
--- Running SE Metric for lookback_rnn ---
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/compute_similarity_error.py --real_list "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt" --real_dir "/scratch1/e20-fyp-x

[INFO] Loading real MIDIs from list: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt (limit=100)
[INFO] Listing generated MIDIs from: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/melody_rnn/output/generated_samples_for_eval
[INFO] Found 100 real, 100 generated files.
[INFO] Computing curve for Real Data...
[INFO] Computing curve for Generated Data...
Similarity Error (SE): 0.256483
[INFO] Saved results to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/se_results_lookback_rnn.json
--- Running SE Metric for museformer_set1 ---
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/compute_similarity_error.py --real_list "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt" --real_dir "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp

[INFO] Loading real MIDIs from list: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt (limit=100)
[INFO] Listing generated MIDIs from: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/museformer_output
[INFO] Found 100 real, 20 generated files.
[INFO] Computing curve for Real Data...
[INFO] Computing curve for Generated Data...
Similarity Error (SE): 0.107722
[INFO] Saved results to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/se_results_museformer_set1.json
--- Running SE Metric for validation ---
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/compute_similarity_error.py --real_list "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt" --real_dir "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/muse

[INFO] Loading real MIDIs from list: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt (limit=100)
[INFO] Listing generated MIDIs from: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/validation_sample_100
[INFO] Found 100 real, 100 generated files.
[INFO] Computing curve for Real Data...
[INFO] Computing curve for Generated Data...
Similarity Error (SE): 0.007730
[INFO] Saved results to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/se_results_validation.json
--- Running SE Metric for model6 ---
Executing: /home/e20365/miniconda3/envs/muspy-env/bin/python /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/compute_similarity_error.py --real_list "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt" --real_dir "/scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/museform

[INFO] Loading real MIDIs from list: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/data/lmd_preprocessed_3/meta/valid.txt (limit=100)
[INFO] Listing generated MIDIs from: /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/notebooks/xLSTM-4-recurrent-state/single-shot-generation/results/gen-after-preprocess-24k-dataset/midi
[INFO] Found 100 real, 120 generated files.
[INFO] Computing curve for Real Data...
[INFO] Computing curve for Generated Data...
Similarity Error (SE): 0.142355
[INFO] Saved results to /scratch1/e20-fyp-xlstm-music-generation/e20fyptemp1/fyp-musicgen/evaluation/results/se_results_model6.json


## 4. Aggregate Results and Compare

In [5]:
def load_summary(csv_path):
    if not csv_path.exists():
        return {}
    df = pd.read_csv(csv_path)
    return dict(zip(df['Metric'], df['Formatted']))

def load_se(json_path):
    if not json_path.exists():
        return "N/A"
    with open(json_path, 'r') as f:
        data = json.load(f)
    return f"{data.get('SE', 0):.4f}"

# Load Data
data_m1 = load_summary(RESULTS_DIR / "musical_summary_model1.csv")
data_m1.update(load_summary(RESULTS_DIR / "ssm_summary_model1.csv"))
data_m1['Similarity Error (SE)'] = load_se(RESULTS_DIR / "se_results_model1.json")

data_m2 = load_summary(RESULTS_DIR / "musical_summary_model2.csv")
data_m2.update(load_summary(RESULTS_DIR / "ssm_summary_model2.csv"))
data_m2['Similarity Error (SE)'] = load_se(RESULTS_DIR / "se_results_model2.json")

data_m3 = load_summary(RESULTS_DIR / "musical_summary_model3.csv")
data_m3.update(load_summary(RESULTS_DIR / "ssm_summary_model3.csv"))
data_m3['Similarity Error (SE)'] = load_se(RESULTS_DIR / "se_results_model3.json")

data_m4 = load_summary(RESULTS_DIR / "musical_summary_model4.csv")
data_m4.update(load_summary(RESULTS_DIR / "ssm_summary_model4.csv"))
data_m4['Similarity Error (SE)'] = load_se(RESULTS_DIR / "se_results_model4.json")

data_m5 = load_summary(RESULTS_DIR / "musical_summary_model5.csv")
data_m5.update(load_summary(RESULTS_DIR / "ssm_summary_model5.csv"))
data_m5['Similarity Error (SE)'] = load_se(RESULTS_DIR / "se_results_model5.json")

data_lookback_rnn = load_summary(RESULTS_DIR / "musical_summary_lookback_rnn.csv")
data_lookback_rnn.update(load_summary(RESULTS_DIR / "ssm_summary_lookback_rnn.csv"))
data_lookback_rnn['Similarity Error (SE)'] = load_se(RESULTS_DIR / "se_results_lookback_rnn.json")

data_museformer = load_summary(RESULTS_DIR / "musical_summary_museformer_set1.csv")
data_museformer.update(load_summary(RESULTS_DIR / "ssm_summary_museformer_set1.csv"))
data_museformer['Similarity Error (SE)'] = load_se(RESULTS_DIR / "se_results_museformer_set1.json")

data_validation = load_summary(RESULTS_DIR / "musical_summary_validation.csv")
data_validation.update(load_summary(RESULTS_DIR / "ssm_summary_validation.csv"))
data_validation['Similarity Error (SE)'] = load_se(RESULTS_DIR / "se_results_validation.json")

data_model6 = load_summary(RESULTS_DIR / "musical_summary_model6.csv")
data_model6.update(load_summary(RESULTS_DIR / "ssm_summary_model6.csv"))
data_model6['Similarity Error (SE)'] = load_se(RESULTS_DIR / "se_results_model6.json")

# Construct Comparison Table
all_metrics = sorted(list(set(data_m1.keys()) | set(data_m2.keys()) | set(data_m3.keys()) | set(data_m4.keys()) | set(data_m5.keys()) | set(data_lookback_rnn.keys()) | set(data_museformer.keys()) | set(data_validation.keys()) | set(data_model6.keys())))

# Prioritize key metrics order
priority = ['Similarity Error (SE)', 'pitch_range', 'key_stability', 'chord_diversity',
            'note_density_mean', 'duration_entropy', 'block_coherence', 'struct_entropy']
remaining = [m for m in all_metrics if m not in priority]
ordered_metrics = [m for m in priority if m in all_metrics] + remaining

comparison_data = []
for m in ordered_metrics:
    comparison_data.append({
        "Metric": m,
        "Model 1 (Checkpoint 84000)": data_m1.get(m, "N/A"),
        "Model 2 (Jan 28 Batch)": data_m2.get(m, "N/A"),
        "Model 3 (Feb 25 Batch)": data_m3.get(m, "N/A"),
        "Model 4 (xLSTM-4 Recurrent)": data_m4.get(m, "N/A"),
        "Model 5 (After Preprocess)": data_m5.get(m, "N/A"),
        "Lookback RNN": data_lookback_rnn.get(m, "N/A"),
        "Museformer": data_museformer.get(m, "N/A"),
        "Validation/Human": data_validation.get(m, "N/A"),
        "xLSTM (24k Dataset)": data_model6.get(m, "N/A")
    })

comp_df = pd.DataFrame(comparison_data)

# Save and Display
comp_df.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
print("\n=== Model Comparison Table ===")
print(comp_df.to_markdown(index=False))
comp_df


=== Model Comparison Table ===


| Metric                | Model 1 (Checkpoint 84000)   | Model 2 (Jan 28 Batch)   | Model 3 (Feb 25 Batch)   | Model 4 (xLSTM-4 Recurrent)   | Model 5 (After Preprocess)   | Lookback RNN      | Museformer        | Validation/Human   | xLSTM (24k Dataset)   |
|:----------------------|:-----------------------------|:-------------------------|:-------------------------|:------------------------------|:-----------------------------|:------------------|:------------------|:-------------------|:----------------------|
| Similarity Error (SE) | 0.0639                       | 0.1352                   | 0.0934                   | 0.0673                        | 0.1424                       | 0.2565            | 0.1077            | 0.0077             | 0.1424                |
| pitch_range           | 38.0000 ± 19.5221            | 61.0000 ± 13.5071        | 28.2405 ± 13.4338        | 52.6917 ± 23.7422             | 51.3417 ± 17.4638            | 18.2800 ± 7.3788  | 57.1000 ± 14.3083 | 48.8000 ±

,Metric,Model 1 (Checkpoint 84000),Model 2 (Jan 28 Batch),Model 3 (Feb 25 Batch),Model 4 (xLSTM-4 Recurrent),Model 5 (After Preprocess),Lookback RNN,Museformer,Validation/Human,xLSTM (24k Dataset)
0,Similarity Error (SE),0.0639,0.1352,0.0934,0.0673,0.1424,0.2565,0.1077,0.0077,0.1424
1,pitch_range,38.0000 ± 19.5221,61.0000 ± 13.5071,28.2405 ± 13.4338,52.6917 ± 23.7422,51.3417 ± 17.4638,18.2800 ± 7.3788,57.1000 ± 14.3083,48.8000 ± 10.7825,51.3417 ± 17.4638
2,key_stability,0.5211 ± 0.2855,0.3191 ± 0.1800,0.6349 ± 0.3000,0.4180 ± 0.2229,0.3653 ± 0.2333,0.3339 ± 0.1765,0.2239 ± 0.1528,0.3577 ± 0.1973,0.3653 ± 0.2333
3,chord_diversity,0.4283 ± 0.2168,0.6562 ± 0.1730,0.2303 ± 0.1948,0.4695 ± 0.2028,0.5592 ± 0.2048,0.4778 ± 0.2870,0.5896 ± 0.1452,0.3010 ± 0.1269,0.5592 ± 0.2048
4,note_density_mean,31.7950 ± 18.3028,32.3524 ± 12.1580,32.6008 ± 19.2353,39.9492 ± 24.9810,27.4110 ± 15.0262,5.9660 ± 2.5348,23.6426 ± 9.7298,39.5787 ± 17.1004,27.4110 ± 15.0262
5,duration_entropy,1.0289 ± 0.5807,1.4158 ± 0.4846,1.0429 ± 0.5584,1.2044 ± 0.5059,1.2532 ± 0.4077,0.8985 ± 0.4084,0.8378 ± 0.7784,1.2629 ± 0.4098,1.2532 ± 0.4077
6,block_coherence,0.3858 ± 0.1871,0.3744 ± 0.1001,0.3845 ± 0.2413,0.3980 ± 0.1611,0.4233 ± 0.1785,0.5616 ± 0.2560,0.3672 ± 0.0864,0.4493 ± 0.1192,0.4233 ± 0.1785
7,struct_entropy,5.3976 ± 1.7005,6.0465 ± 1.2837,6.1431 ± 1.1436,6.1783 ± 1.6454,6.5158 ± 1.5908,6.1834 ± 0.8686,7.3524 ± 1.2831,8.0216 ± 0.7243,6.5158 ± 1.5908
8,avg_offdiag,0.6305 ± 0.2097,0.5293 ± 0.1157,0.7297 ± 0.2178,0.5840 ± 0.1638,0.5015 ± 0.1662,0.4012 ± 0.1441,0.3750 ± 0.1486,0.5392 ± 0.1197,0.5015 ± 0.1662
9,bars_N,31.5875 ± 19.2398,39.0000 ± 17.1622,39.1519 ± 18.2891,48.6417 ± 40.2847,60.6917 ± 57.0797,47.0000 ± 16.0806,94.4500 ± 56.1834,93.6100 ± 31.3010,60.6917 ± 57.0797
